In [6]:
!apt-get update
!apt-get install -y portaudio19-dev

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [99.9 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.5 MB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,309 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:14 http://s

In [7]:
!pip install pyaudio faster-whisper numpy

  Using cached PyAudio-0.2.14.tar.gz (47 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached faster_whisper-1.2.1-py3-none-any.whl.metadata (16 kB)
  Using cached ctranslate2-4.8.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached onnxruntime-1.27.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.4 kB)
  Using cached av-18.0.0-cp311-abi3-manylinux_2_28_x86_64.whl.metadata (4.5 kB)
Using cached faster_whisper-1.2.1-py3-none-any.whl (1.1 MB)
Using cached av-18.0.0-cp311-abi3-manylinux_2_28_x86_64.whl (35.5 MB)
Using cached ctranslate2-4.8.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (39.5 MB)
Using cached onnxruntime-1.27.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (18.7 MB)
  Created wheel for pyaudio: filename=pyaudio-0.2.14-cp312-cp312-linux_x86_64.whl size=68670 sha256=0b035b218a117cf

In [8]:
!pip install PyAudio==0.2.11

  Using cached PyAudio-0.2.11.tar.gz (37 kB)
  Preparing metadata (setup.py) ... done
  Created wheel for PyAudio: filename=PyAudio-0.2.11-cp312-cp312-linux_x86_64.whl size=52929 sha256=af7f0cfea32208ad05558aed2845d3c5d5c3047eb1ea65e1f5b68fd1df4b84c3
  Stored in directory: /root/.cache/pip/wheels/65/3b/dd/f89390a284b9042070707568809def73ce4f3ecdd116e74fd8
Successfully built PyAudio
  Attempting uninstall: PyAudio
    Found existing installation: PyAudio 0.2.14
    Uninstalling PyAudio-0.2.14:
      Successfully uninstalled PyAudio-0.2.14


In [10]:
import pyaudio
import numpy as np
import queue
from faster_whisper import WhisperModel

# ==========================================
# 1. 오디오 및 큐(Queue) 설정
# ==========================================
RATE = 16000        # Whisper 모델이 요구하는 기본 샘플링 레이트 (16kHz)
CHUNK = 16000       # 1초 단위로 끊어서 처리 (16000 프레임 = 1초)
FORMAT = pyaudio.paInt16
CHANNELS = 1

# I/O 병목을 막기 위한 스레드 간 데이터 버퍼(Queue)
audio_queue = queue.Queue()

# ==========================================
# 2. PyAudio 콜백 함수 (생산자 역할)
# ==========================================
# 마이크에서 소리가 들어올 때마다 백그라운드 스레드에서 자동으로 실행됨
def audio_callback(in_data, frame_count, time_info, status):
    # 캡처된 바이트 데이터를 즉시 큐에 집어넣음 (I/O 블로킹 방지)
    audio_queue.put(in_data)
    return (None, pyaudio.paContinue)

# ==========================================
# 3. 모델 초기화
# ==========================================
print("모델을 로드하는 중입니다... (최초 실행 시 다운로드)")
# GPU(CUDA)를 사용할 수 있다면 'cuda', 아니면 'cpu' 사용
# compute_type="float16" (GPU) 또는 "int8" (CPU)로 최적화
# model = WhisperModel("small", device="cuda", compute_type="float16")
model = WhisperModel("small", device="cpu", compute_type="int8")
print("모델 로드 완료!")

# ==========================================
# 4. 오디오 스트림 시작
# ==========================================
p = pyaudio.PyAudio()
stream = p.open(format=FORMAT,
                channels=CHANNELS,
                rate=RATE,
                input=True,
                frames_per_buffer=CHUNK,
                stream_callback=audio_callback)

print("🎙️ 실시간 음성 인식을 시작합니다. (종료하려면 Ctrl+C)")
stream.start_stream()

# ==========================================
# 5. 메인 추론 루프 (소비자 역할)
# ==========================================
try:
    while True:
        # 큐에서 1초 분량의 오디오 조각(Chunk)을 꺼냄
        # 큐가 비어있으면 데이터가 들어올 때까지 대기(Block)함
        in_data = audio_queue.get()

        # Whisper 모델은 16kHz의 float32 형태의 NumPy 배열(-1.0 ~ 1.0)을 입력으로 받음
        audio_data = np.frombuffer(in_data, dtype=np.int16).astype(np.float32) / 32768.0

        # 모델 추론 진행
        # task="transcribe" (해당 언어로 자막 생성)
        # task="translate" (무조건 영어로 번역하여 출력)
        segments, info = model.transcribe(
            audio_data,
            beam_size=5,
            task="translate", # 번역을 원할 경우. 자막만 원하면 "transcribe"
            language="ko"     # 입력 언어 (자동 감지하려면 None)
        )

        # 결과 즉시 출력
        for segment in segments:
            # 텍스트가 비어있지 않은 경우에만 화면에 Print
            text = segment.text.strip()
            if text:
                print(f"[{info.language}] {text}")

except KeyboardInterrupt:
    print("\n종료를 요청하셨습니다. 리소스를 정리합니다.")
finally:
    stream.stop_stream()
    stream.close()
    p.terminate()

모델을 로드하는 중입니다... (최초 실행 시 다운로드)
모델 로드 완료!


OSError: [Errno -9996] Invalid input device (no default output device)